# A2 · The hospital simulator

**Goal:** build the hidden rule, generate 8 months of visits, and check the simulated hospital behaves like a real OPD *before* we trust it.

1. Write `expected_minutes` and `sample_noise` in [slotpred/world.py](../slotpred/world.py) (`TODO(you)`)
2. Run the tests until they pass
3. Generate the data
4. Sanity checks (given)
5. **The oracle** (`TODO(you)`): the best score any model could get

⚠️ We look only at the clean `world` here. The messy export (`raw`) is for A3, so don't explore it yet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from slotpred.config import ROOT, TEST_START
from slotpred import world as w

## Concept: the noise, visualised

How a symmetric bell curve Z becomes right-skewed noise e^Z, and why the 80th percentile carries straight across (0.196 → e^0.196 = 1.217). Explained in [notes/A02-simulator.md](../notes/A02-simulator.md). This cell doesn't need your TODOs, so it runs before the tests pass.

In [ ]:
from scipy.stats import norm

sigma = w.NOISE_SIGMA
mu = -sigma**2 / 2                 # -0.039: makes the noise average exactly 1
z80 = norm.ppf(0.8)                # 0.8416, the z-score for 80%
q80_Z = mu + z80 * sigma           # 0.196, the 80th percentile of Z
q80_noise = np.exp(q80_Z)          # 1.217, the 80th percentile of the noise

z = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 500)
pdf_z = norm.pdf(z, mu, sigma)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Z is a symmetric bell curve
ax1.plot(z, pdf_z)
ax1.fill_between(z, pdf_z, where=z <= q80_Z, alpha=0.3, label="80% of draws")
ax1.axvline(mu, ls=":", color="grey", label=f"centre μ = {mu:.3f}")
ax1.axvline(q80_Z, color="red", label=f"80th percentile = {q80_Z:.3f}")
ax1.annotate("", xy=(q80_Z, pdf_z.max() * 0.5), xytext=(mu, pdf_z.max() * 0.5), arrowprops=dict(arrowstyle="->"))
ax1.text((mu + q80_Z) / 2, pdf_z.max() * 0.54, "0.8416 × σ", ha="center")
ax1.set_title("Steps 1–2: Z ~ Normal(μ, σ), symmetric"); ax1.set_xlabel("Z"); ax1.legend(fontsize=8)

# 2. The conversion e^Z: 0.196 on the x-axis becomes 1.217 on the y-axis
ax2.plot(z, np.exp(z))
ax2.plot([q80_Z, q80_Z], [0, q80_noise], "r--"); ax2.plot([z.min(), q80_Z], [q80_noise, q80_noise], "r--")
ax2.scatter([q80_Z], [q80_noise], color="red", zorder=3)
ax2.text(q80_Z + 0.03, q80_noise - 0.15, f"e^{q80_Z:.3f} = {q80_noise:.3f}", color="red")
ax2.set_ylim(0, np.exp(z.max())); ax2.set_title("Step 3: convert with noise = e^Z")
ax2.set_xlabel("Z"); ax2.set_ylabel("noise multiplier")

# 3. The noise is right-skewed
x = np.exp(z)
pdf_x = pdf_z / x                  # density after the change of variable (dz/dx = 1/x)
ax3.plot(x, pdf_x)
ax3.fill_between(x, pdf_x, where=x <= q80_noise, alpha=0.3, label="the same 80% of draws")
ax3.axvline(np.exp(mu), ls=":", color="grey", label=f"median = {np.exp(mu):.3f}")
ax3.axvline(1.0, ls="--", color="green", label="mean = 1.000")
ax3.axvline(q80_noise, color="red", label=f"P80 = {q80_noise:.3f}")
ax3.set_title("Result: noise multiplier, right-skewed"); ax3.set_xlabel("noise multiplier (× expected minutes)")
ax3.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Check the maths with a million random draws
draws = np.exp(np.random.default_rng(0).normal(mu, sigma, 1_000_000))
print(f"simulated mean   {draws.mean():.3f}   (expect 1.000)")
print(f"simulated median {np.median(draws):.3f}   (expect {np.exp(mu):.3f})")
print(f"simulated P80    {np.quantile(draws, 0.8):.3f}   (expect {q80_noise:.3f})")

## Step 2 - run the tests

Red means not right yet; read the failing test's name and message. Green means done. Rerun this cell after each edit to `world.py`.

In [ ]:
!"{sys.executable}" -m pytest "{ROOT / 'tests' / 'test_world.py'}" -q

## Step 3 - generate the data

In [ ]:
from slotpred import simulate as sim

world, raw = sim.main()
world.head()

## Step 4 - sanity checks

### 4a. Is it right-skewed, like real consults?

In [ ]:
y = world["actual_minutes"]
print(f"mean {y.mean():.1f}  median {y.median():.1f}  P80 {y.quantile(0.8):.1f}  max {y.max():.0f}")

plt.figure(figsize=(8, 3.5))
plt.hist(y, bins=100, range=(0, 60))
plt.axvline(y.median(), color="black", linestyle=":", label="median")
plt.axvline(y.mean(), color="black", linestyle="--", label="mean")
plt.xlabel("actual minutes"); plt.ylabel("visits"); plt.legend(); plt.show()

### 4b. Do departments and visit types come out in the right order?

Compare with the base table. The medians will differ a little: reports, interpreters, age and noise all shift them.

In [ ]:
world.pivot_table(index="department", columns="visit_type", values="actual_minutes", aggfunc="median").round(1)

### 4c. Can we see the drift?

Weekly median for Dermatology vs everyone else. Something should change on 1 July.

In [ ]:
weekly = (world.assign(week=pd.to_datetime(world["date"]).dt.to_period("W").dt.start_time,
                       group=np.where(world["department"] == "Dermatology", "Dermatology", "Others"))
              .groupby(["week", "group"])["actual_minutes"].median().unstack())

weekly.plot(figsize=(9, 3.5), marker=".")
plt.axvline(pd.Timestamp(TEST_START), color="red", linestyle="--", label="1 July")
plt.ylabel("weekly median minutes"); plt.legend(); plt.show()

## Step 5 - TODO(you): the oracle

The oracle knows the hidden rule exactly (`expected_minutes`), but not the noise. Its best predictions are:

- **P50** = expected × e^(−σ²/2)
- **P80** = expected × e^(−σ²/2 + 0.8416·σ)

On the **test period** (`date >= TEST_START`), compute:

| | Oracle | Fixed 15-min slot (today's rule) |
|---|---|---|
| MAE (P50 guess vs actual) | ? | use 15 as the guess |
| Coverage (actual ≤ P80) | ? | actual ≤ 15 |
| Average idle minutes (slot − actual, when positive) | ? | ? |

For the oracle's slot, use P80 **rounded up to 5 minutes** (`np.ceil(p80 / 5) * 5`). Use `w.NOISE_SIGMA` for σ.

In [ ]:
test = world[world["date"] >= TEST_START].copy()

# TODO(you): oracle P50, P80 and slot columns on `test`, then the metrics for both rows of the table.


## Questions - write your answers here

1. What's the oracle's MAE? This is the **irreducible error**: no model can beat it on average.

   *Answer:*

2. Is the oracle's coverage close to 80%? It isn't exactly 80% - why? (Hint: the slot is rounded up to 5 minutes.)

   *Answer:*

3. How does the fixed 15-minute slot compare on coverage and idle time? Which kind of visit does it hurt most?

   *Answer:*

4. (From the lesson) Why does σ = 0.28 make P80 about 27% bigger than the median, whatever the expected minutes are?

   *Answer:*